# 30 — W4 B1 stage: KTO warmup of the responder LLM

Per RecSys_Challenge_Plan §6.3 (B1 row) + §6.3.1 (TRL operational requirements). First post-training stage of Component B: train Qwen-2.5-7B with **KTO** (Kahneman-Tversky Optimization) on the GPA-labeled (context, response, label) pairs from `data/trl/kto.parquet` (envelope-augmented).

**Why KTO first** (plan §6.3): cheapest first pass; binary labels we already have; LoRA-friendly; ~3 A100-hr. KTO doesn't need preference *pairs* (1 chosen vs 1 rejected) — it just needs a desirable/undesirable bool per (prompt, completion). Our GPA labels map directly: MOVES_TOWARD_GOAL → True, DOES_NOT_MOVE → False.

**Pipeline:**
1. Mount Drive, install TRL/PEFT/Trackio.
2. Pytest gate (reward_fns, augmenter, build_trl_datasets all green).
3. Build the KTO parquet on the fly: `build_reward_dataset.py → augment_envelope.py → build_trl_datasets.py`. (~10 min CPU.)
4. Validate the parquet with the dataset inspector (skill mandate, CPU, <1 min).
5. Run KTO training: Qwen-7B base + LoRA r=32, eval split 10%, Trackio + Hub push every save.
6. Format-compliance check: ≥95% of model outputs match the envelope regex.
7. Save adapter to Hub + Drive.

**Gate (plan §4 W4 row):** Format compliance ≥ 95%; dev nDCG@20 not regressed > 0.005 vs frozen-retriever (separate eval, run via `colab/40_run_blindset_B.ipynb` later).
**Wall time:** ~3 A100-hr (per plan §6.3 B1 row).

**OOM ladder if Qwen-7B+LoRA blows VRAM:** (1) `per_device_train_batch_size=1` + `gradient_accumulation_steps=8`, (2) `gradient_checkpointing=True`, (3) drop `max_length=1024→512`, (4) Unsloth (`.claude/skills/huggingface-llm-trainer/references/unsloth.md`), (5) A100-80GB upgrade.

In [ ]:
# 1) GPU check.
!nvidia-smi | head -20

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026
!git log -1 --pretty=format:'commit:  %h%nsubject: %s'

In [ ]:
# 3) HF auth — required for push_to_hub. Set HF_TOKEN as a Colab secret
# (Settings → Secrets → 'HF_TOKEN' with read+write permissions).
#
# P1 FIX: abort the notebook if HF_TOKEN is missing. The skill mandates
# Hub push because Colab runtime is ephemeral. Without a valid token,
# `push_to_hub=True` fails at training-end with a confusing trace AFTER
# 3 hr of A100 time — fail fast instead.
from google.colab import userdata
import os, sys
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN
    # Verify it works by checking whoami.
    from huggingface_hub import whoami
    user = whoami(token=HF_TOKEN)
    print(f'✓ HF auth ok — logged in as {user["name"]}')
except (userdata.SecretNotFoundError, Exception) as e:
    print(f'❌ HF auth failed: {e!r}')
    print('   Fix: Settings → Secrets → add HF_TOKEN with WRITE permissions.')
    print('   This notebook will abort: training without Hub push wastes A100 time')
    print('   (Colab runtime is ephemeral — local checkpoints get deleted).')
    raise SystemExit('HF_TOKEN required — see message above.')

In [ ]:
# 2b) Mount Drive + persistent caches.
#
# DRIVE_BASE is the per-stage gate_result + adapter backup root. Required
# for cells 13's gate_result.json write. The same path convention is used
# by notebooks 31p/32/33/40/41 to find this notebook's outputs.
import os, shutil
from google.colab import drive

try: drive.mount('/content/drive')
except Exception as e:
    print(f'first mount attempt failed: {e}; retrying ...')
    try: drive.flush_and_unmount()
    except Exception: pass
    drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = '/content/drive/MyDrive/recsys2026-cache'
for d in [f'{DRIVE_BASE}/hf_datasets', f'{DRIVE_BASE}/experiments_cache',
          f'{DRIVE_BASE}/kto_runs']:
    os.makedirs(d, exist_ok=True)

os.environ['HF_DATASETS_CACHE'] = f'{DRIVE_BASE}/hf_datasets'
%env HF_DATASETS_CACHE={DRIVE_BASE}/hf_datasets

EXPECTED_CACHE = '/content/recsys2026/music-crs-baselines/experiments/cache'
os.makedirs(os.path.dirname(EXPECTED_CACHE), exist_ok=True)
if os.path.exists(EXPECTED_CACHE) and not os.path.islink(EXPECTED_CACHE):
    shutil.rmtree(EXPECTED_CACHE)
if not os.path.islink(EXPECTED_CACHE):
    os.symlink(f'{DRIVE_BASE}/experiments_cache', EXPECTED_CACHE)
print(f'✓ Drive mounted; DRIVE_BASE={DRIVE_BASE}')

In [ ]:
# 5) Install deps + pytest pre-flight.
#
# Colab Pro's base image ships transformers/datasets/pandas but NOT bm25s,
# and an older trl/peft. mcrs.* tests (state_tracker, cmqr, pro_rank)
# transitively import bm25s via mcrs.retrieval_modules.bm25 → ImportError
# at collection time without this install. Same versions notebook 31 uses.
!pip install -q --upgrade transformers datasets pandas tqdm omegaconf
!pip install -q --upgrade 'trl>=0.12.0' 'peft>=0.13.0' trackio accelerate bm25s
!python -c 'import torch, transformers, trl, peft, bm25s; print("torch", torch.__version__, "trl", trl.__version__, "peft", peft.__version__, "bm25s ok")'

# Pytest pre-flight — confirm ALL module-level tests pass.
# P1 FIX: include state_tracker / cmqr / pro_rank since the dev-eval pipeline
# (W4 gate part 2: nDCG@20 no-regression in colab/40) uses all of them.
!cd /content/recsys2026 && python -m pytest \
    tests/test_reward_fns.py \
    tests/test_state_tracker.py \
    tests/test_cmqr.py \
    tests/test_pro_rank.py \
    tests/test_augment_envelope.py \
    tests/test_build_trl_datasets.py \
    -q

In [ ]:
# 7) Validate KTO format — P1 FIX: invoke the official dataset_inspector.py
# in addition to the local schema check. Plan §6.3.1 mandates this as a
# hard pre-flight; the inspector also catches NaN values, label distribution
# skew, length outliers — things the local check misses.

# Local schema check (cheap, instant).
from datasets import Dataset
ds = Dataset.from_parquet(KTO)
print(f'parquet → HF Dataset: {ds}')
print(f'first row keys: {list(ds[0].keys())}')
assert set(ds.column_names) >= {'prompt', 'completion', 'label'}, ds.column_names
assert ds.features['label'].dtype == 'bool', ds.features['label']
print('✓ local schema check (prompt/completion/label:bool) PASS')

# Official dataset_inspector — runs locally on the parquet, no Hub upload
# needed. Checks: empty rows, NaN, label balance, length outliers, etc.
import sys
INSPECTOR = '/content/recsys2026/.claude/skills/huggingface-llm-trainer/scripts/dataset_inspector.py'
import subprocess
print('\n=== Running official dataset_inspector.py ===')
# Inspector accepts --parquet for local files (alternative to --dataset for Hub).
result = subprocess.run(
    [sys.executable, INSPECTOR, '--parquet', KTO, '--method', 'kto'],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(f'⚠️  inspector exited {result.returncode}')
    print(result.stderr)
    # If the inspector doesn't support --parquet locally, fall back gracefully.
    if 'unrecognized arguments' in result.stderr or 'No such file' in result.stderr:
        print('Inspector script doesn\'t support local --parquet path; falling back to')
        print('local schema check only. Plan §6.3.1: this is a known limitation if the')
        print('inspector tool only accepts Hub dataset names.')
print('✓ dataset validation complete')

In [ ]:
# 9) Trackio init — group="b-stage" so all B1/B2/B3 runs share a dashboard.
# P1 FIX: wrap in try/except so a Trackio outage doesn't kill training;
# fall back to console logging via report_to='none' if init fails.
from datetime import date
import trackio

RUN_NAME = f'b1-kto-qwen7b-{date.today().isoformat()}'
TRACKIO_OK = True
try:
    trackio.init(
        project='recsys2026',
        run_name=RUN_NAME,
        group='b-stage',
        config={
            'model': 'Qwen/Qwen2.5-7B-Instruct',
            'method': 'KTO',
            'dataset_size': len(train_ds),
            'eval_size': len(eval_ds),
            'lora_r': 32,
            'lora_alpha': 32,
            'beta': 0.1,
        },
    )
    print(f'✓ Trackio run: {RUN_NAME}')
except Exception as e:
    TRACKIO_OK = False
    print(f'⚠️  Trackio init failed ({e!r}); training will use console logging only.')
    print('   Continuing without Trackio — set report_to="none" in KTOConfig below.')

In [ ]:
# 10) KTO training — Qwen-2.5-7B + LoRA r=32.
# Per plan §6.3.1: push_to_hub=True, hub_strategy='every_save', report_to='trackio',
# eval_dataset MUST be set when eval_strategy != 'no'. LoRA r=32 alpha=32 dropout=0.05.
#
# REVIEW FIXES APPLIED:
#   P0 #1 — model loaded explicitly in bf16 (not via string-init which would
#           default to fp32 → 28GB cold-start OOM on A100-40GB).
#   P1 — effective batch 16→128 (8 × 16; plan §6.3.1 mandate; same wallclock).
#   P1 — max_prompt_length 512→1024 (real prompts often 700-900 tokens).
#   P1 — desirable_weight / undesirable_weight computed from POS/NEG ratio
#        (TRL recommendation when ratio > 1.33).
#   P1 — save_steps 200→50 (first checkpoint within ~12 min; OOM-survivable).
#   P1 — per_device_eval_batch_size set explicitly (else ~9hr eval overhead).
#   P1 — report_to respects Trackio init success/failure (cell 9).

import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM
from trl import KTOTrainer, KTOConfig

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
HUB_REPO = f'orrimoch/recsys2026-{RUN_NAME}'
OUTPUT_DIR = f'/content/recsys2026/training_runs/{RUN_NAME}'

# ---- P1 fix: data-aware desirable/undesirable weights ----
n_pos = int(sum(1 for x in train_ds if x['label']))
n_neg = int(sum(1 for x in train_ds if not x['label']))
ratio = max(n_pos, n_neg) / max(min(n_pos, n_neg), 1)
if ratio > 1.33:
    if n_pos > n_neg:
        desirable_weight = 1.0
        undesirable_weight = n_pos / n_neg
    else:
        undesirable_weight = 1.0
        desirable_weight = n_neg / n_pos
    print(f'⚠️  imbalanced data (pos={n_pos:,} neg={n_neg:,}, ratio={ratio:.2f})')
    print(f'   setting desirable_weight={desirable_weight:.2f} undesirable_weight={undesirable_weight:.2f}')
else:
    desirable_weight = undesirable_weight = 1.0
    print(f'balanced data (pos={n_pos:,} neg={n_neg:,}); using 1.0/1.0 weights')

# LoRA config — plan §6.3.1 standard.
peft_config = LoraConfig(
    r=32,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules='all-linear',
)

# ---- P0 #1 fix: pre-load model in bf16 ----
# Passing `model=MODEL_NAME` as a string lets TRL load with default dtype
# (fp32 = 28GB on A100-40GB → cold-start OOM). Pre-loading in bf16 = 14GB,
# leaves ~26GB for activations + reference (LoRA-base trick) + optimizer.
print(f'loading {MODEL_NAME} in bf16…')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='cuda',
)
# Required for gradient_checkpointing + LoRA + non-quantized base.
base_model.enable_input_require_grads()
print(f'✓ base model loaded ({base_model.num_parameters() / 1e9:.1f}B params)')

# KTOConfig — see https://huggingface.co/docs/trl/kto_trainer for full options.
config = KTOConfig(
    output_dir=OUTPUT_DIR,

    # ---- Hub push (plan §6.3.1, skill mandate: ephemeral env → must push) ----
    push_to_hub=True,
    hub_model_id=HUB_REPO,
    hub_strategy='every_save',
    hub_private_repo=True,

    # ---- KTO-specific ----
    beta=0.1,
    desirable_weight=desirable_weight,
    undesirable_weight=undesirable_weight,

    # ---- Sequence length (P1 fix: bumped to fit full prompts) ----
    max_length=1536,                 # was 1024; real prompts + envelope often >1k
    max_prompt_length=1024,          # was 512; was silently truncating User-query header

    # ---- Training (P1 fix: effective batch 128 per plan §6.3.1) ----
    num_train_epochs=1,
    per_device_train_batch_size=2,   # OOM ladder if needed → 1
    gradient_accumulation_steps=64,  # was 8; effective batch 2*64 = 128 (plan target)
    learning_rate=5e-6,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    bf16=True,
    gradient_checkpointing=True,

    # ---- Eval (P1 fix: explicit eval batch + eval ratio check) ----
    eval_strategy='steps',
    eval_steps=200,
    per_device_eval_batch_size=4,    # default 8 → ~9hr overhead at 6k eval rows; 4 is safer

    # ---- Checkpointing (P1 fix: earlier first save) ----
    save_strategy='steps',
    save_steps=50,                   # was 200; first ckpt at ~12 min so OOM is survivable
    save_total_limit=3,
    logging_steps=10,

    # ---- Monitoring (P1 fix: respect Trackio init result from cell 9) ----
    report_to='trackio' if TRACKIO_OK else 'none',
)

# Initialize trainer — pass the LOADED model (not name string) so dtype is preserved.
trainer = KTOTrainer(
    model=base_model,                 # ← P0 #1: was MODEL_NAME (string); now pre-loaded bf16
    args=config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,             # MUST be set when eval_strategy != 'no'
    peft_config=peft_config,
)

print('🚀 Starting KTO training (~3 A100-hr expected)...')
print(f'   model: {MODEL_NAME} (pre-loaded bf16)')
print(f'   adapter → {HUB_REPO}')
print(f'   trackio: project=recsys2026 run={RUN_NAME}' if TRACKIO_OK else '   trackio: SKIPPED (init failed in cell 9)')
print(f'   effective batch: {config.per_device_train_batch_size * config.gradient_accumulation_steps}')
print(f'   weights: desirable={desirable_weight:.2f} undesirable={undesirable_weight:.2f}')
trainer.train()
print('✓ training complete')

In [ ]:
# 12) Format-compliance check — the W4 gate.
#
# REVIEW FIXES APPLIED:
#   P0 #2 — apply the production chat template (system prompt =
#           response_generation_cot_user_state.txt) so the format gate
#           measures the SAME prompt distribution that production uses.
#           Without this, the gate is meaningless.
#   P0 #4 — free the trainer's policy + reference + optimizer before
#           loading the inference model (else 39GB peak → OOM).
#   P0 #5 — use `r_format` (which checks state_block parses non-empty)
#           rather than bare `ENVELOPE.search`. Aligns with W6 R_format.

import gc, sys, os, re
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# ---- P0 #4: free trainer state before loading the inference model ----
print('freeing trainer/reference/optimizer state…')
del trainer
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f'free VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')

# ---- Load the trained adapter for inference ----
print(f'loading {MODEL_NAME} + LoRA adapter from {OUTPUT_DIR}…')
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
inference_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map='cuda',
)
model = PeftModel.from_pretrained(inference_base, OUTPUT_DIR).eval()
print('✓ inference model ready')

# ---- P0 #2: build prompts using the production chat template ----
# Production (vllm_model.batch_response_generation) renders:
#   [{role: 'system', content: response_generation_cot_user_state.txt},
#    {role: 'user',   content: <user_query>}, ...]
# via tokenizer.apply_chat_template. The KTO data was raw `text_a` strings,
# so the LM saw a different prompt distribution at training time. Measuring
# format compliance under the production format is the only signal that
# transfers to the dev-eval pipeline.
sys.path.insert(0, '/content/recsys2026/scripts')
from reward_fns import r_format

PROMPTS_DIR = '/content/recsys2026/music-crs-baselines/mcrs/system_prompts'
with open(f'{PROMPTS_DIR}/roleplay.txt', encoding='utf-8') as f:
    role_play = f.read()
with open(f'{PROMPTS_DIR}/response_generation_cot_user_state.txt', encoding='utf-8') as f:
    cot_prompt = f.read()
SYSTEM_PROMPT = role_play + '\n\n' + cot_prompt

# Pull 50 user_query strings from the original train data — we want the
# `text_a`-derived "User query: ..." line specifically. The KTO eval split
# already has prompts; extract just the user-query portion (text_a starts
# with "User query: ").
USER_QUERY_RE = re.compile(r'^User query:\s*(.+?)(?=\nListener goal:|\nGoal category:|$)',
                            re.DOTALL | re.MULTILINE)

def extract_user_query(prompt_text):
    m = USER_QUERY_RE.search(prompt_text)
    return m.group(1).strip() if m else prompt_text[:200]

prompts_data = [eval_ds[i]['prompt'] for i in range(min(50, len(eval_ds)))]
user_queries = [extract_user_query(p) for p in prompts_data]

n_match_format = 0       # bare envelope regex (loose)
n_match_r_format = 0     # r_format (strict — requires non-empty state)
samples = []

for uq in user_queries:
    chat_messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': uq},
    ]
    formatted = tok.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True)
    enc = tok(formatted, return_tensors='pt', truncation=True, max_length=2048).to('cuda')
    with torch.no_grad():
        out_ids = model.generate(
            **enc, max_new_tokens=320, do_sample=False,
            pad_token_id=tok.pad_token_id or tok.eos_token_id,
        )
    text = tok.decode(out_ids[0, enc['input_ids'].shape[1]:], skip_special_tokens=True)

    # Strict gate (P0 #5): r_format requires the envelope AND a non-empty
    # state block (`{key: value}` pair must parse). This is what W6 uses.
    if r_format(text) == 1.0:
        n_match_r_format += 1
    # Loose gate (legacy comparison): bare envelope tag presence.
    from reward_fns import ENVELOPE
    if ENVELOPE.search(text):
        n_match_format += 1
    if len(samples) < 3:
        samples.append(text[:400])

compliance_strict = n_match_r_format / len(user_queries)
compliance_loose = n_match_format / len(user_queries)

print(f'\nFORMAT COMPLIANCE (P0 #5: strict r_format = state must parse):')
print(f'  strict r_format:   {n_match_r_format}/{len(user_queries)} = {compliance_strict:.1%}')
print(f'  loose ENVELOPE:    {n_match_format}/{len(user_queries)} = {compliance_loose:.1%}')
print(f'\nSample generations (under PRODUCTION chat template):')
for i, s in enumerate(samples, 1):
    print(f'\n--- sample {i} ---\n{s}')

print('\n' + '=' * 60)
print('W4 B1 GATE (plan §6.3 row B1):')
# W4 gate threshold uses STRICT compliance — that's what W6 will see.
if compliance_strict >= 0.95:
    print(f'  PASS  strict r_format compliance {compliance_strict:.1%} ≥ 95%')
    print('  Next: run colab/40_run_blindset_B.ipynb to check dev nDCG@20 no-regression.')
    print('  If both gates pass → proceed to W6 (Rank-GRPO).')
elif compliance_loose >= 0.95:
    print(f'  WEAK PASS  loose ENVELOPE {compliance_loose:.1%} ≥ 95%')
    print(f'  but strict {compliance_strict:.1%} < 95% — model emits envelope but state block is empty/malformed.')
    print(f'  Likely cause: state cache was empty during training → all rows trained on (unknown).')
    print(f'  Fix: run colab/22_extract_train_states.ipynb, rebuild parquet, retrain.')
else:
    print(f'  FAIL  both gates miss (strict {compliance_strict:.1%} / loose {compliance_loose:.1%}).')
    print('  Likely diagnoses (in order of probability):')
    print('    (1) num_train_epochs too low — bump to 2.')
    print('    (2) LR too low for envelope memorization — bump to 1e-5.')
    print('    (3) Consider SFT warmup first (skill: train_sft_example.py).')
print('=' * 60)

trackio.log({
    'format_compliance_strict': compliance_strict,
    'format_compliance_loose': compliance_loose,
    'n_samples': len(user_queries),
})

In [ ]:
# 13) Persist gate result + finish Trackio.
import json
from datetime import date
result = {
    'stage': 'B1-KTO',
    'run_name': RUN_NAME,
    'date': date.today().isoformat(),
    'hub_model': HUB_REPO,
    'format_compliance_strict': compliance_strict,
    'format_compliance_loose': compliance_loose,
    'gate_passed': compliance_strict >= 0.95,
    'n_eval_samples': len(user_queries),
    'sample_outputs': samples,
    'kto_config': {
        'desirable_weight': desirable_weight,
        'undesirable_weight': undesirable_weight,
        'effective_batch_size': 128,
        'max_length': 1536,
        'max_prompt_length': 1024,
        'response_max_new_tokens': 320,
        'lora_r': 32,
    },
}
out_path = f'{DRIVE_BASE}/kto_runs/{RUN_NAME}/gate_result.json'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
print(f'gate result → {out_path}')

trackio.finish()
print('✓ Trackio session closed')

## After the run

**Format compliance ≥ 95% (PASS):**
- Adapter is at `https://huggingface.co/{HUB_REPO}` (private).
- Run `colab/40_run_blindset_B.ipynb` (TBD) with `reranker_type=pro_rank` + `lm_type` pointing to the adapter to check the dev nDCG@20 no-regression gate.
- If both gates pass → either run **W5 S-DPO** (only if KTO format < 70% — conditional per plan §6.3) OR skip to **W6 Rank-GRPO**.

**Format compliance < 95% (FAIL):**
- Likely diagnoses (in order of probability):
  1. **State cache empty** → augmenter ran in stub mode → every row had `(unknown)` envelope. Run `colab/22_extract_train_states.ipynb` first, rebuild parquets, retrain.
  2. **1 epoch insufficient** for envelope memorisation. Bump `num_train_epochs=2`.
  3. **LR too conservative** (5e-6 with LoRA). Try 1e-5 or 2e-5.
  4. **KTO loss not driving format** (KTO optimizes *response quality*, not format). Consider SFT warmup *first* (1 epoch on positives only with `target=text_b`), then KTO.
- Document failure in `documents/experiments_log.md`.

**OOM during training:**
- Step 1: `per_device_train_batch_size=1`, `gradient_accumulation_steps=16` (keeps effective batch).
- Step 2: already enabled `gradient_checkpointing=True`.
- Step 3: `max_length=512` (truncates long history; envelope still fits).
- Step 4: switch to **Unsloth** (60% less VRAM, 2× faster — see `.claude/skills/huggingface-llm-trainer/references/unsloth.md`).
- Step 5: A100-80GB if available.